# HRC Safety Log — huấn luyện phát hiện bất thường

Notebook này dùng dữ liệu hai cảm biến siêu âm đã ghi từ ESP32. Mô hình Isolation Forest học vùng hoạt động SAFE; ngưỡng khoảng cách vẫn là lớp bảo vệ quyết định cho EmergencyStop.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ai_model.feature_engineering import load_recordings, FEATURE_COLUMNS
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib, json

In [ ]:
data = load_recordings(ROOT / 'ai_model' / 'data' / 'raw')
data[['label'] + FEATURE_COLUMNS].head(), data['label'].value_counts()

In [ ]:
X = data[FEATURE_COLUMNS]
y = data['label_id']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
normal = X_train[y_train == 0]
model = Pipeline([('scale', StandardScaler()), ('detector', IsolationForest(n_estimators=150, contamination=0.08, random_state=42))])
model.fit(normal)
predicted = (model.predict(X_test) == -1).astype(int)
actual = (y_test > 0).astype(int)
print(classification_report(actual, predicted, zero_division=0))
print(confusion_matrix(actual, predicted))

In [ ]:
artifact = {'model': model, 'feature_columns': FEATURE_COLUMNS, 'thresholds_cm': {'warning': 60.0, 'danger': 30.0}, 'training_rows': int(len(normal))}
(ROOT / 'ai').mkdir(exist_ok=True)
joblib.dump(artifact, ROOT / 'ai' / 'model.joblib')
print('Saved ai/model.joblib')

## Diễn giải cho báo cáo

Đây là anomaly detection bán giám sát: nhãn SAFE dùng để học phân bố bình thường, còn APPROACHING/DANGER/RETREATING dùng để đánh giá. Blockchain không lưu toàn bộ dữ liệu cảm biến; gateway tạo mã định danh sự kiện và hash, sau đó ghi hash cùng mức độ cảnh báo lên chain để chứng minh nhật ký chưa bị sửa.